In [29]:
pip list

Package            Version
------------------ ------------
aiobotocore        2.26.0
aiohappyeyeballs   2.6.1
aiohttp            3.13.4
aioitertools       0.13.0
aiosignal          1.4.0
altgraph           0.17.2
appnope            0.1.4
asttokens          3.0.1
async-timeout      5.0.1
attrs              26.1.0
botocore           1.41.5
certifi            2026.2.25
charset-normalizer 3.4.6
comm               0.2.3
cramjam            2.11.0
debugpy            1.8.20
decorator          5.2.1
duckdb             1.4.4
exceptiongroup     1.3.1
executing          2.2.1
fastparquet        2024.11.0
frozenlist         1.8.0
fsspec             2025.10.0
future             0.18.2
greenlet           3.2.5
idna               3.11
importlib_metadata 8.7.1
ipykernel          6.31.0
ipython            8.18.1
jedi               0.19.2
jmespath           1.1.0
jupyter_client     8.6.3
jupyter_core       5.8.1
macholib           1.15.2
matplotlib-inline  0.2.1
multidict          6.7.1
nest-asyncio     

In [30]:
pip install requests pandas duckdb python-dotenv s3fs

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [31]:
# import requests
# import pandas as pd
# import time
# from io import BytesIO
# from datetime import datetime

# import sys
# sys.path.append('..')
# from utils.bucket_utils import get_duck_con, get_fs

In [32]:
# บังคับให้ Jupyter โหลดไฟล์ .py ใหม่ทุกครั้งที่มีการแก้ไข
# %load_ext autoreload
# %autoreload 2

# import requests
# import pandas as pd
# import time
# from io import BytesIO
# from datetime import datetime
# import sys

# # ชี้ Path ให้มองเห็นโฟลเดอร์ utils
# sys.path.append('..')

# # Import ฟังก์ชันจากไฟล์ของเรา
# from utils.bucket_utils import get_duck_con, get_fs

# # เรียกใช้งานเพื่อทดสอบการเชื่อมต่อ
# con = get_fs()
# print("✅ Import และเชื่อมต่อสำเร็จ!")

In [33]:
import os
!{sys.executable} -m pip install playwright pandas pyarrow fastparquet
!{sys.executable} -m playwright install chromium

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [34]:
import asyncio
from playwright.async_api import async_playwright, TimeoutError
import pandas as pd
import os
from datetime import datetime

In [35]:
def get_today_thai_format():
    now = datetime.now()
    thai_months = ["มกราคม", "กุมภาพันธ์", "มีนาคม", "เมษายน", "พฤษภาคม", "มิถุนายน",
                   "กรกฎาคม", "สิงหาคม", "กันยายน", "ตุลาคม", "พฤศจิกายน", "ธันวาคม"]
    thai_year = now.year + 543
    return f"{now.day:02d} {thai_months[now.month - 1]} {thai_year}"

async def scrape_pathum_all_districts():
    keywords = [
        "เมืองปทุมธานี", "คลองหลวง", "ธัญบุรี", "ลำลูกกา", "สามโคก", "ลาดหลุมแก้ว", "หนองเสือ", "รังสิต",
        "ถนนกาญจนาภิเษก", "ถนนพหลโยธิน", "อุดรรัถยา", "ถนนรังสิตนครนายก", "ติวานนท์", "รังสิตปทุมธานี", 
        "เชียงราก", "เลียบคลองเปรมประชากร", "ไสวประชาราษฎร์", "คลอง 1", "คลอง 2", "คลอง 3", "คลอง 4", "คลอง 5"
    ]
    all_data = []
    today_real_date = get_today_thai_format() 

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, args=["--disable-blink-features=AutomationControlled"])
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
        page = await context.new_page()

        for keyword in keywords:
            print(f"🔍 กำลังดึงข้อมูลพื้นที่: {keyword}...")
            try:
                await page.goto("https://www.js100.com/en/site/home/search_advance", wait_until="domcontentloaded", timeout=30000)
                await page.wait_for_timeout(1000)
                
                search_input = await page.wait_for_selector('input[name="search_text"], #search_result_input', timeout=10000)
                await search_input.click()
                await search_input.fill(keyword)
                await page.wait_for_timeout(500)
                await search_input.press("Enter")
                
                await page.wait_for_selector('#search_result_list li', timeout=20000)
                await page.wait_for_timeout(1000)
                
                items = await page.query_selector_all('#search_result_list li')
                for item in items:
                    h4_tag = await item.query_selector('h4')
                    raw_datetime = (await h4_tag.inner_text()).strip() if h4_tag else ""
                    date_str, time_str = "ไม่ระบุ", "ไม่ระบุ"
                    if "," in raw_datetime:
                        parts = raw_datetime.split(",")
                        date_str, time_str = parts[0].strip(), parts[1].strip()
                    else:
                        date_str = raw_datetime
                    
                    if date_str == "วันนี้": 
                        date_str = today_real_date
                    
                    a_tag, p_tag = await item.query_selector('a'), await item.query_selector('p')
                    category, title, details = "", "", ""
                    if a_tag and (await a_tag.inner_text()).strip():
                        category, title = "ข่าว/ประกาศ", (await a_tag.inner_text()).strip()
                        link = await a_tag.get_attribute('href')
                        details = f"https://www.js100.com{link}" if link and link.startswith('/') else link
                    elif p_tag:
                        category, title, details = "รายงานจราจร", f"อัปเดตจราจรพื้นที่ {keyword}", (await p_tag.inner_text()).strip()
                    else:
                        category, title, details = "อื่นๆ", "ไม่มีหัวข้อ", (await item.inner_text()).replace(raw_datetime, "").strip()

                    all_data.append({"พื้นที่ (คำค้น)": keyword, "วันที่": date_str, "เวลา": time_str, "ประเภท": category, "หัวข้อ": title, "รายละเอียด": details})
            
            except TimeoutError:
                print(f"   ⚠️ ข้ามพื้นที่ '{keyword}' (ใช้เวลาโหลดนานเกินไป หรือไม่พบข้อมูล)")
                continue
            except asyncio.CancelledError:
                print(f"\n🛑 การทำงานถูกยกเลิกกลางคัน! (อาจเผลอปิดเบราว์เซอร์ หรือกดหยุด)")
                break
            except Exception as e:
                print(f"   ❌ เกิดข้อผิดพลาดกับ '{keyword}': {e}")
                continue

        await browser.close()
    return pd.DataFrame(all_data)

In [36]:
async def main():
    print("🚀 เริ่มกระบวนการดึงข้อมูล...")
    df_new = await scrape_pathum_all_districts()

    now = datetime.now()
    today_real_date = get_today_thai_format() 
    timestamp = now.strftime("%Y%m%d_%H%M")   

    desktop_path = os.path.expanduser("~/Desktop")
    target_folder = os.path.join(desktop_path, "JS100")

    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print(f"📁 สร้างโฟลเดอร์ใหม่ที่: {target_folder}")

    if not df_new.empty:
        # 1. กรองเอาเฉพาะของวันนี้
        df_today = df_new[df_new['วันที่'] == today_real_date].copy()
        
        # 2. 💡 ลบข้อมูลที่ซ้ำซ้อนกัน (เช็คจาก วันที่, เวลา, หัวข้อ และรายละเอียด)
        # ใช้ keep='first' เพื่อเก็บข้อมูลที่ค้นเจอครั้งแรกไว้ และลบตัวที่ซ้ำออก
        df_today = df_today.drop_duplicates(subset=['วันที่', 'เวลา', 'หัวข้อ', 'รายละเอียด'], keep='first')
        
        if df_today.empty:
            print(f"⚠️ ไม่พบรายการของวันที่ {today_real_date} (ข้ามการบันทึก)")
            return pd.DataFrame()
        else:
            file_name = os.path.join(target_folder, f"JS100_Pathum_{timestamp}.parquet")
            
            df_today.to_parquet(file_name, engine='fastparquet', index=False)
            
            print(f"✅ บันทึกข้อมูลสำเร็จ! (กรองข้อมูลซ้ำออกแล้ว)")
            print(f"📄 ชื่อไฟล์: JS100_Pathum_{timestamp}.parquet")
            print(f"📍 ที่อยู่: {file_name}")
            print("✅ กระบวนการเสร็จสิ้น!")
            
            # ส่งตารางออกมาให้ใช้งานต่อ
            return df_today
    else:
        print("🤷‍♂️ ไม่พบข้อมูลใหม่จากการค้นหาครั้งนี้")
        return pd.DataFrame()

if __name__ == "__main__":
    # บันทึกข้อมูลที่ได้ลงตัวแปร table_today
    table_today = await main()

🚀 เริ่มกระบวนการดึงข้อมูล...
🔍 กำลังดึงข้อมูลพื้นที่: เมืองปทุมธานี...
🔍 กำลังดึงข้อมูลพื้นที่: คลองหลวง...
   ⚠️ ข้ามพื้นที่ 'คลองหลวง' (ใช้เวลาโหลดนานเกินไป หรือไม่พบข้อมูล)
🔍 กำลังดึงข้อมูลพื้นที่: ธัญบุรี...
🔍 กำลังดึงข้อมูลพื้นที่: ลำลูกกา...
   ⚠️ ข้ามพื้นที่ 'ลำลูกกา' (ใช้เวลาโหลดนานเกินไป หรือไม่พบข้อมูล)
🔍 กำลังดึงข้อมูลพื้นที่: สามโคก...
🔍 กำลังดึงข้อมูลพื้นที่: ลาดหลุมแก้ว...
🔍 กำลังดึงข้อมูลพื้นที่: หนองเสือ...
🔍 กำลังดึงข้อมูลพื้นที่: รังสิต...
   ⚠️ ข้ามพื้นที่ 'รังสิต' (ใช้เวลาโหลดนานเกินไป หรือไม่พบข้อมูล)
🔍 กำลังดึงข้อมูลพื้นที่: ถนนกาญจนาภิเษก...
🔍 กำลังดึงข้อมูลพื้นที่: ถนนพหลโยธิน...
🔍 กำลังดึงข้อมูลพื้นที่: อุดรรัถยา...
🔍 กำลังดึงข้อมูลพื้นที่: ถนนรังสิตนครนายก...
   ⚠️ ข้ามพื้นที่ 'ถนนรังสิตนครนายก' (ใช้เวลาโหลดนานเกินไป หรือไม่พบข้อมูล)
🔍 กำลังดึงข้อมูลพื้นที่: ติวานนท์...
🔍 กำลังดึงข้อมูลพื้นที่: รังสิตปทุมธานี...
🔍 กำลังดึงข้อมูลพื้นที่: เชียงราก...
🔍 กำลังดึงข้อมูลพื้นที่: เลียบคลองเปรมประชากร...
🔍 กำลังดึงข้อมูลพื้นที่: ไสวประชาราษฎร์...
🔍 กำลังดึงข้อมูลพื้นท

In [37]:
display(table_today)

""


In [38]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

# 1. โหลด Environment Variables
load_dotenv()
raw_endpoint = os.environ.get("GCS_ENDPOINT_URL", "")
key = os.environ.get("GCS_ACCESS_KEY")
secret = os.environ.get("GCS_SECRET_KEY")

# 2. ทำความสะอาด Endpoint (DuckDB S3 ไม่ต้องการ https://)
clean_endpoint = raw_endpoint.replace("https://", "").replace("http://", "").rstrip("/")

# 3. สร้าง Connection
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# 4. สร้าง Secret แบบ S3 (เพื่อให้ยัด ENDPOINT เข้าไปได้)
con.execute(f"""
    CREATE OR REPLACE SECRET my_gcs_s3_secret (
        TYPE S3,
        KEY_ID '{key}',
        SECRET '{secret}',
        ENDPOINT '{clean_endpoint}',
        URL_STYLE 'path' 
    );
""")
print("✅ ตั้งค่า Connection (S3 Compatibility Mode) สำเร็จ!")

✅ ตั้งค่า Connection (S3 Compatibility Mode) สำเร็จ!


In [39]:
df = table_today 

# 1. กำหนด Path (กลับมาใช้ s3:// ตามโครงสร้างเดิมของทีม)
path = 's3://pea-oms/test.parquet'

print(f"⏳ กำลังอัปโหลดข้อมูลไปที่ {path}...")

# 2. อัปโหลดข้อมูลลง Bucket
con.execute(f"""
    COPY (
        SELECT *
        FROM df
    )
    TO '{path}' (FORMAT parquet)
""")
print("✅ Upload สำเร็จ!")

print("-" * 30)
print("⏳ กำลังทดสอบดึงข้อมูลกลับมา...")

# 3. ทดสอบอ่านข้อมูล
result_df = con.execute(f"""
    SELECT * FROM read_parquet('{path}')       
""").df()

print("✅ Read สำเร็จ! ข้อมูลที่ได้:")
display(result_df)

⏳ กำลังอัปโหลดข้อมูลไปที่ s3://pea-oms/test.parquet...


InvalidInputException: Invalid Input Error: Need a DataFrame with at least one column

In [ ]:
import requests
import pandas as pd
import time
from io import BytesIO
from datetime import datetime
import sys

# ชี้ Path ให้มองเห็นโฟลเดอร์ utils
sys.path.append('..')

# Import ฟังก์ชันจากไฟล์ของเรา
from utils.bucket_utils import get_duck_con, get_fs

# เรียกใช้งานเพื่อทดสอบการเชื่อมต่อ
con = get_fs()
print("✅ Import และเชื่อมต่อสำเร็จ!")

✅ Import และเชื่อมต่อสำเร็จ!


In [ ]:
# import pandas as pd
# from gcs_connection import get_duckdb_s3_connection

# print("⏳ กำลังเชื่อมต่อกับ Storage ขององค์กร...")
# con = get_duckdb_s3_connection()
# print("✅ เชื่อมต่อสำเร็จ!")

In [ ]:
df = table_today 

print("ตารางข้อมูลที่เตรียมพร้อมสำหรับอัปโหลด:")
display(df)

NameError: name 'table_today' is not defined

In [ ]:
# กำหนด Path ปลายทาง (ใช้ s3:// ตามที่องค์กรกำหนด)
path = 's3://pea-oms/test_js100.parquet'

print(f"⏳ กำลังอัปโหลดข้อมูลไปที่ {path}...")

# สั่งอัปโหลดข้อมูล
con.execute(f"""
    COPY (
        SELECT *
        FROM df
    )
    TO '{path}' (FORMAT parquet)
""")

print("✅ Upload สำเร็จ!")

⏳ กำลังอัปโหลดข้อมูลไปที่ s3://pea-oms/test.parquet...


CatalogException: Catalog Error: Table with name df does not exist!
Did you mean "pg_depend"?

LINE 4:         FROM df
                     ^

In [ ]:
result_df = con.execute(f"""
    SELECT * FROM read_parquet('{path}')       
""").df()

display(result_df)

,พื้นที่ (คำค้น),วันที่,เวลา,ประเภท,หัวข้อ,รายละเอียด
0,คลองหลวง,27 มีนาคม 2569,10:36น.,รายงานจราจร,อัปเดตจราจรพื้นที่ คลองหลวง,อุบัติเหตุ ถนนกาญจนาภิเษก จาก ต่างระดับธัญบุรี...
1,ธัญบุรี,27 มีนาคม 2569,10:36น.,รายงานจราจร,อัปเดตจราจรพื้นที่ ธัญบุรี,อุบัติเหตุ ถนนกาญจนาภิเษก จาก ต่างระดับธัญบุรี...
2,ถนนกาญจนาภิเษก,27 มีนาคม 2569,16:36น.,รายงานจราจร,อัปเดตจราจรพื้นที่ ถนนกาญจนาภิเษก,อุบัติเหตุ ถนนพหลโยธิน ขาเข้า จาก แยกวังน้อย ม...
3,ถนนกาญจนาภิเษก,27 มีนาคม 2569,11:15น.,รายงานจราจร,อัปเดตจราจรพื้นที่ ถนนกาญจนาภิเษก,ถนนกาญจนาภิเษก จาก ต่างระดับรามคำแหง มุ่งหน้า ...
4,ถนนกาญจนาภิเษก,27 มีนาคม 2569,10:36น.,รายงานจราจร,อัปเดตจราจรพื้นที่ ถนนกาญจนาภิเษก,อุบัติเหตุ ถนนกาญจนาภิเษก จาก ต่างระดับธัญบุรี...
5,ถนนกาญจนาภิเษก,27 มีนาคม 2569,09:18น.,รายงานจราจร,อัปเดตจราจรพื้นที่ ถนนกาญจนาภิเษก,อุบัติเหตุ ถนนกาญจนาภิเษก ขาเข้า จาก ตรงข้ามหม...
6,ถนนกาญจนาภิเษก,27 มีนาคม 2569,03:53น.,รายงานจราจร,อัปเดตจราจรพื้นที่ ถนนกาญจนาภิเษก,อุบัติเหตุ ถนนกาญจนาภิเษก ช่วงโรงเรียนเลิศหล้า...
7,ถนนพหลโยธิน,27 มีนาคม 2569,16:36น.,รายงานจราจร,อัปเดตจราจรพื้นที่ ถนนพหลโยธิน,อุบัติเหตุ ถนนพหลโยธิน ขาเข้า จาก แยกวังน้อย ม...
